# 01. Try and except

**Tier:** Intermediate. **Why you'd use this:** A score field arrives as text. Most rows are integers. A few are blank or junk. You want the good scores as numbers and a clear stand-in for the bad ones, without crashing the whole import.

**After this lesson you can:** catch one exception type and keep going; explain EAFP versus checking first; avoid a bare `except` that swallows `KeyboardInterrupt`.
About 25 minutes. **Uses:** [Definitions](../05-functions/01-definitions-lesson.ipynb)


## Quick review

1. What does a function return when it never hits `return`?
2. Why is a mutable default list shared across calls?

<details><summary>Answers</summary>

It returns `None`. The default object is created once at `def` time.
</details>


## The idea

`try` runs a block. If that block raises a matching exception, `except` runs instead of crashing. Python style is EAFP: easier to ask forgiveness than permission. Try the operation, catch the failure. LBYL, look before you leap, checks first with `if`. EAFP wins when the check can race the real operation, or when the check duplicates work the operation already does.

Catch the type you mean. `except ValueError` handles bad text. A bare `except:` also catches `KeyboardInterrupt` and `SystemExit`, which you almost never want to silence.

<details><summary>Explain more</summary>

`except Exception` is narrower than bare `except`, but still wide. Prefer the specific type. An except that returns a stand-in turns a crash into a value the caller can test.
</details>


## See it work

Read a score from text. Bad text becomes `None`.


In [1]:
def read_score(text):
    # 1. Try the conversion that usually works.
    try:
        return int(text)
    # 2. Only this failure is normal for junk text.
    except ValueError:
        return None


print(read_score("12"), read_score("na"), read_score(""))

12 None None


## Check yourself

Answer these before you run the cell.

1. What does `read_score("3.5")` return?
2. Which except is too wide, and what does it hide?


In [2]:
def read_score(text):
    try:
        return int(text)
    except ValueError:
        return None


print(read_score("3.5"))


def wide(text):
    try:
        return int(text)
    except Exception:
        return None


print(wide("na"))
print(issubclass(KeyboardInterrupt, Exception))
print(issubclass(KeyboardInterrupt, BaseException))

None
None
False
True


<details><summary>Answers</summary>

`"3.5"` is not an int literal, so `ValueError` becomes `None`. `except Exception` catches almost every error in your code. `KeyboardInterrupt` is not an `Exception`, but a bare `except:` would catch it, because it sits under `BaseException`.
</details>


## Why it works this way

An exception is an object that unwinds frames until an `except` matches its type. Matching uses the same subclass rules as `isinstance`. `ValueError` is an `Exception`. `KeyboardInterrupt` is a `BaseException` but not an `Exception`. A bare `except:` catches `BaseException`, so Ctrl-C becomes a silent `None` if you return from that handler. That is why narrow catches are the rule. Deep dive: [Memory management](../../internals/11-memory/01-refcount-lesson.ipynb).

EAFP is preferred because the happy path is one clear block, and the failure path names the real error. LBYL with `isdigit` rejects `"-3"` and still fails to catch every way `int` can raise. Checking `if right != 0` before a division also races in concurrent code: another thread can change `right` after the check. The `try` around the real division does not have that gap.

The cost of a `try` that does not raise is tiny. The cost of a raised exception is building a traceback with every active frame. Use exceptions for unusual failure, not for ordinary looping. The hierarchy is the design: library code raises specific types, and callers choose how wide to catch. Catching `Exception` is a last resort around a boundary, not the default inside a helper. The `__mro__` cell shows where `KeyboardInterrupt` sits: under `BaseException`, beside `Exception`, never under it.

A useful mental test: if silencing this error would hide a bug in your own code, the catch is too wide. `ValueError` from `int` is data. `TypeError` from calling a missing function is a bug. They should not share one handler that returns `None`.


In [3]:
print(ValueError.__mro__)
print(KeyboardInterrupt.__mro__)

(<class 'ValueError'>, <class 'Exception'>, <class 'BaseException'>, <class 'object'>)
(<class 'KeyboardInterrupt'>, <class 'BaseException'>, <class 'object'>)


## When to use it

Use `try`/`except` when failure is unusual and the operation itself is the check. Use an `if` when the condition is cheap and is not what the operation would raise. Catch the specific type. Don't use bare `except`. Don't use exceptions to hop out of ordinary loops.


## Common mistake

A bare `except` that returns a stand-in. It looks helpful, and it also swallows bugs you meant to see, including a Ctrl-C on some paths if you wrote `except:` with no type.


In [4]:
def read_score_wrong(text):
    try:
        return int(text)
    except:
        return None


print(read_score_wrong("na"))

None


Name the exception. `ValueError` is the one `int` raises for bad text. Other failures still surface.


In [5]:
def read_score(text):
    try:
        return int(text)
    except ValueError:
        return None


print(read_score("na"))

None


## Try it

Change `read_score` so an empty string returns `0` and other bad text still returns `None`. Which check belongs outside the `try`, and which failure still needs `except`?


## Recap

- `try`/`except` turns a specific failure into a handled path.
- EAFP tries the operation. Narrow catches keep real bugs loud.
- Bare `except` is too wide.

**Next:** [practice](01-try-except-practice.ipynb)
